# All Known Human Genes — HGNC Complete Set to CSV

Downloads the **HGNC (HUGO Gene Nomenclature Committee) complete gene set** — the authoritative reference for approved human gene symbols and names — and writes it out as a clean CSV.

- Source: https://www.genenames.org/download/statistics-and-files/
- ~43,000 gene records (protein-coding genes, RNA genes, pseudogenes, and other loci)
- Updated by HGNC every Tuesday and Friday

**Notes:**
- Requires outbound internet access to `storage.googleapis.com` (HGNC's current host) or `ftp.ebi.ac.uk` (older mirror, used as fallback).
- The file is a single flat TSV — no zip/unzip needed, unlike the FDA drug data.
- Includes withdrawn/merged symbols if you don't filter on `status`; set `APPROVED_ONLY = True` below to keep only currently approved symbols.
- If you only want the gene symbol list (no metadata), see the last cell for a bare single-column export.

In [1]:
import pandas as pd

PRIMARY_URL = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt"
FALLBACK_URL = "https://ftp.ebi.ac.uk/pub/databases/genenames/hgnc/tsv/hgnc_complete_set.txt"

APPROVED_ONLY = True  # keep only status == "Approved" (drops withdrawn/merged symbols)
OUTPUT_CSV = "hgnc_all_human_genes.csv"
OUTPUT_SYMBOLS_ONLY_CSV = "hgnc_gene_symbols_only.csv"

## 1. Download the complete gene set

In [2]:
try:
    df = pd.read_csv(PRIMARY_URL, sep="\t", low_memory=False)
    print(f"Downloaded from primary URL: {PRIMARY_URL}")
except Exception as e:
    print(f"Primary URL failed ({e}), trying fallback...")
    df = pd.read_csv(FALLBACK_URL, sep="\t", low_memory=False)
    print(f"Downloaded from fallback URL: {FALLBACK_URL}")

print(f"Loaded {len(df)} records, {len(df.columns)} columns")
df.head()

Downloaded from primary URL: https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt
Loaded 45045 records, 54 columns


,hgnc_id,symbol,name,locus_group,locus_type,status,location,location_sortable,alias_symbol,alias_name,...,cd,lncrnadb,enzyme_id,intermediate_filament_db,rna_central_id,lncipedia,gtrnadb,agr,mane_select,gencc
0,HGNC:5,A1BG,alpha-1-B glycoprotein,protein-coding gene,gene with protein product,Approved,19q13.43,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HGNC:5,ENST00000263100.8|NM_130786.4,NaN
1,HGNC:37133,A1BG-AS1,A1BG antisense RNA 1,non-coding RNA,"RNA, long non-coding",Approved,19q13.43,NaN,FLJ23569,NaN,...,NaN,NaN,NaN,NaN,URS00007E4F6E,A1BG-AS1,NaN,HGNC:37133,NaN,NaN
2,HGNC:24086,A1CF,APOBEC1 complementation factor,protein-coding gene,gene with protein product,Approved,10q11.23,NaN,ACF|ASP|ACF64|ACF65|APOBEC1CF,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HGNC:24086,ENST00000373997.8|NM_014576.4,NaN
3,HGNC:7,A2M,alpha-2-macroglobulin,protein-coding gene,gene with protein product,Approved,12p13.31,NaN,FWP007|S863-7|CPAMD5,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HGNC:7,ENST00000318602.12|NM_000014.6,HGNC:7
4,HGNC:27057,A2M-AS1,A2M antisense RNA 1,non-coding RNA,"RNA, long non-coding",Approved,12p13.31,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,URS00001F234A,A2M-AS1,NaN,HGNC:27057,NaN,NaN


## 2. Filter and select useful columns

The full file has 50+ columns (cross-references to Entrez, Ensembl, UCSC, UniProt, RefSeq, etc.). Keeping the most broadly useful ones here — adjust `KEEP_COLS` if you want the full mapping table instead.

In [3]:
if APPROVED_ONLY and "status" in df.columns:
    df = df[df["status"] == "Approved"].copy()
    print(f"Filtered to approved-only: {len(df)} records")

KEEP_COLS = [
    "hgnc_id", "symbol", "name", "locus_group", "locus_type",
    "status", "location", "alias_symbol", "prev_symbol",
    "gene_group", "entrez_id", "ensembl_gene_id", "refseq_accession",
]
keep_cols = [c for c in KEEP_COLS if c in df.columns]
df_out = df[keep_cols].sort_values("symbol").reset_index(drop=True)
df_out.head(20)

Filtered to approved-only: 45045 records


,hgnc_id,symbol,name,locus_group,locus_type,status,location,alias_symbol,prev_symbol,gene_group,entrez_id,ensembl_gene_id,refseq_accession
0,HGNC:5,A1BG,alpha-1-B glycoprotein,protein-coding gene,gene with protein product,Approved,19q13.43,NaN,NaN,Immunoglobulin like domain containing,1.0,ENSG00000121410,NM_130786
1,HGNC:37133,A1BG-AS1,A1BG antisense RNA 1,non-coding RNA,"RNA, long non-coding",Approved,19q13.43,FLJ23569,NCRNA00181|A1BGAS|A1BG-AS,Antisense RNAs,503538.0,ENSG00000268895,NR_015380
2,HGNC:24086,A1CF,APOBEC1 complementation factor,protein-coding gene,gene with protein product,Approved,10q11.23,ACF|ASP|ACF64|ACF65|APOBEC1CF,NaN,RNA binding motif containing,29974.0,ENSG00000148584,NM_014576
3,HGNC:7,A2M,alpha-2-macroglobulin,protein-coding gene,gene with protein product,Approved,12p13.31,FWP007|S863-7|CPAMD5,NaN,Alpha-2-macroglobulin family,2.0,ENSG00000175899,NM_000014
4,HGNC:27057,A2M-AS1,A2M antisense RNA 1,non-coding RNA,"RNA, long non-coding",Approved,12p13.31,NaN,NaN,Antisense RNAs,144571.0,ENSG00000245105,NR_026971
5,HGNC:23336,A2ML1,alpha-2-macroglobulin like 1,protein-coding gene,gene with protein product,Approved,12p13.31,FLJ25179|p170,CPAMD9,Alpha-2-macroglobulin family,144568.0,ENSG00000166535,NM_144670
6,HGNC:41022,A2ML1-AS1,A2ML1 antisense RNA 1,non-coding RNA,"RNA, long non-coding",Approved,12p13.31,NaN,NaN,Antisense RNAs,100874108.0,ENSG00000256661,NR_046715
7,HGNC:41523,A2ML1-AS2,A2ML1 antisense RNA 2,non-coding RNA,"RNA, long non-coding",Approved,12p13.31,NaN,NaN,Antisense RNAs,106478979.0,ENSG00000256904,NaN
8,HGNC:30005,A3GALT2,"alpha 1,3-galactosyltransferase 2",protein-coding gene,gene with protein product,Approved,1p35.1,IGBS3S|IGB3S,A3GALT2P,Glycosyltransferase family 6,127550.0,ENSG00000184389,NM_001080438
9,HGNC:18149,A4GALT,"alpha 1,4-galactosyltransferase (P1PK blood gr...",protein-coding gene,gene with protein product,Approved,22q13.2,A14GALT|Gb3S|P(k),P1,"Alpha 1,4-glycosyltransferases|Blood group ant...",53947.0,ENSG00000128274,NM_017436


## 3. Write out the full CSV

In [4]:
df_out.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(df_out)} rows to {OUTPUT_CSV}")

Wrote 45045 rows to hgnc_all_human_genes.csv


## 4. (Optional) Bare gene-symbol-only list

In [ ]:
symbols_only = df_out[["symbol"]].drop_duplicates().sort_values("symbol")
symbols_only.to_csv(OUTPUT_SYMBOLS_ONLY_CSV, index=False)
print(f"Wrote {len(symbols_only)} gene symbols to {OUTPUT_SYMBOLS_ONLY_CSV}")